# 一定程度上的并行

来自视频 [徒手实现深度循环神经网络--大语言模型的雏形](https://www.bilibili.com/video/BV1uS421o7ts)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from Tools.scripts.combinerefs import combine
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
%matplotlib inline


torch.manual_seed(12046)

In [2]:
# 一些超参数
learning_rate = 1e-3
eval_iters = 10
batch_size=1000
sequence_len=64
# 如果有GPU，该脚本将使用GPU进行计算
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
datasets = load_dataset('json', data_files='./datasets/python/final/jsonl/train/*.jsonl.gz')
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])

In [4]:
class CharTokenizer:

    def __init__(self, data, end_ind=0):
        # data: list[str]
        # 得到所有的字符
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in enumerate(chars)}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in self.char2ind.items()}
        self.end_ind = end_ind

    def encode(self, x):
        # x: str
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        # x: int or list[x]
        if isinstance(x, int):
            return self.ind2char[x]
        return [self.ind2char[i] for i in x]

tokenizer = CharTokenizer(datasets['original_string'])
test_str = 'def f(x):'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

[70, 71, 72, 2, 72, 10, 90, 11, 28]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~ö'

In [5]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)


    def forward(self, input, hidden=None):
        # input: (B, T, C)  # T 是序列的长度
        # hidden: (B, H)  # H 是隐藏层的大小
        # out: (B, T, H)

        B, T, C = input.shape
        re = []

        if hidden is None:
            hidden = self.init_hidden(B, input.device)

        # 沿着第二维的维度进行循环，并同时处理 B 次
        for i in range(T):
            # 要求序列一定要为 T 长
            combined = torch.concat((input[:, i, :], hidden), dim=-1)  # 以 , 左右没每一个维度，第一维全部取出，第二维度选择第 i 个元素，第三位全部取出，这里的意思是在每一个维度中，都取出第 i 个内容，并且第 i 个内容不变，因为只有三维可以理解为横着切了一刀  (B, C + H)
            hidden = F.relu(self.i2h(combined))  # (B, H)
            re.append(hidden)

        return torch.stack(re, dim=1)  # (B, T, H)

    def init_hidden(self, B, device):
        return torch.zeros((B, self.hidden_size)).to(device)



In [11]:
### torch stack 展示
a = torch.zeros(3, 4)
b = a + 1
c = torch.stack([a, b], dim=1)

print(a)
print(b)
print(c)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[[0., 0., 0., 0.],
         [1., 1., 1., 1.]],

        [[0., 0., 0., 0.],
         [1., 1., 1., 1.]],

        [[0., 0., 0., 0.],
         [1., 1., 1., 1.]]])


In [12]:
r = RNN(3, 4)
x = torch.randn(5, 2, 3)
r(x).shape

torch.Size([5, 2, 4])

In [20]:
### 数据填充：需要填充/需要截断
### 层归一化：https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
class CharRNNBatch(nn.Module):

    def __init__(self, vs):
        super().__init__()
        emb_size = 256
        hidden_size = 128
        self.emb = nn.Embedding(vs, emb_size)
        self.rnn1 = RNN(emb_size, hidden_size)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.rnn2 = RNN(hidden_size, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.lm = nn.Linear(hidden_size, vs)
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x: (B, T)
        # 暂不实现初始隐藏状态的输入
        B = x.shape[0]
        embeddings = self.emb(x)  # (B, T, emb_size)
        h = F.relu(self.ln1(self.rnn1(embeddings)))  # (B, T, hidden_size)
        h = self.dp(h)
        h = F.relu(self.ln2(self.rnn2(h)))  # (B, T, hidden_size)
        h = self.dp(h)
        out = self.lm(h)
        return out


In [21]:
c_model = CharRNNBatch(len(tokenizer.char2ind)).to(device)
c_model

CharRNNBatch(
  (emb): Embedding(98, 256)
  (rnn1): RNN(
    (i2h): Linear(in_features=384, out_features=128, bias=True)
  )
  (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (rnn2): RNN(
    (i2h): Linear(in_features=256, out_features=128, bias=True)
  )
  (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lm): Linear(in_features=128, out_features=98, bias=True)
  (dp): Dropout(p=0.4, inplace=False)
)

In [22]:
@torch.no_grad()
def generate(model, context, tokenizer, max_new_tokens=300):
    # context: (1, T)
    #out = []
    out = context.tolist()[0]
    model.eval()
    for _ in range(max_new_tokens):
        #可以考虑截断背景，使得文本生成更加贴近训练
        #logits = model(context[:, -sequence_len:])
        logits = model(context)            # (1, T, 98)
        probs = F.softmax(logits[:, -1, :], dim=-1)  # (1, 98)
        # 随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  # (1, 1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [23]:
context = torch.tensor(tokenizer.encode('def'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

def*ZO(5/o|("YP{BoE ?|uw=3:1'L$?Q9NN[-K|=CK|AM:iKca"|+Q3-<sA*gWS$ö0NG!q9T3"y~m5-a)'W~]\rm&B"%{r
c"i,k^DAx1zk@}@*N
L"jIT^~JuciGPi&.Qp!)_a_GB_*zC!la#,p=84WVJk%ycbyJ{sXK$>cYtd"c!E&/zG^K~>A8'N]^~Di"~]"/N/5N!^-iVo6ZMa`ösTM'>#)C,n4JNP
pZ\[DKWFp`
"omF^2drBZT08]byU%+bk7pc^&GG%0E$bm{og$O5+:0_JT49/EpB\roW&\JnJuk


In [25]:
def process(data, tokenizer, sequence_len=sequence_len):
    text = data['original_string']
    # text is list[str]
    inputs, labels = [], []
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        # 有bug，无法处理长度过小的数据
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i: i + sequence_len])
            labels.append(enc[i + 1: i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}

In [27]:
# 将数据分为训练集和测试集
tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)

f = lambda x: process(x, tokenizer)
tokenized = tokenized.map(f, batched=True, remove_columns=datasets.column_names)
tokenized.set_format(type='torch', device=device)

In [28]:
train_loader = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=batch_size, shuffle=True)

In [30]:
next(iter(train_loader))

{'inputs': tensor([[ 2,  2, 76,  ..., 73,  2, 31],
         [71,  2, 81,  ..., 81,  2, 85],
         [67, 86, 31,  ..., 77, 16, 85],
         ...,
         [10, 10, 85,  ..., 85, 71, 78],
         [10, 61, 19,  ..., 65, 84, 71],
         [20, 18, 19,  ..., 74, 71, 84]], device='cuda:0'),
 'labels': tensor([[ 2, 76, 69,  ...,  2, 31,  2],
         [ 2, 81, 72,  ...,  2, 85, 86],
         [86, 31, 48,  ..., 16, 85, 83],
         ...,
         [10, 85, 71,  ..., 71, 78, 72],
         [61, 19, 14,  ..., 84, 71, 79],
         [18, 19, 23,  ..., 71, 84,  1]], device='cuda:0')}

In [31]:
def estimate_loss(model):
    re = {}
    # 将模型切换至评估模式
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['test'] = _loss(model, test_loader)
    # 将模型切换至训练模式
    model.train()
    return re

@torch.no_grad()
def _loss(model, data_loader):
    """
    计算模型在不同数据集下面的评估指标
    """
    loss = []
    data_iter= iter(data_loader)
    # 随机使用多个批量数据来预估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None)
        if data is None:
            data_iter = iter(data_loader)
            data = next(data_iter, None)
        inputs, labels = data['inputs'], data['labels']  # (B, T)
        logits = model(inputs)                           # (B, T, vs)
        # 请参考官方文档
        loss.append(F.cross_entropy(logits.transpose(-2, -1), labels).item())
    return torch.tensor(loss).mean().item()

estimate_loss(c_model)

{'train': 4.72510290145874, 'test': 4.7315497398376465}

In [32]:
def train_model(model, optimizer, epochs=10):
    # 记录模型在训练集上的模型损失
    lossi = []
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels']  # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                           # (B, T, vs)
            loss = F.cross_entropy(logits.transpose(-2, -1), labels)
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats["train"]:.4f}'
        test_loss = f'test loss {stats["test"]:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [33]:
### PyTorch的RNN实现：https://pytorch.org/docs/stable/generated/torch.nn.RNN.html
l = train_model(c_model, optim.Adam(c_model.parameters(), lr=learning_rate))

epoch  0: train loss 1.4161, test loss 1.5274
epoch  1: train loss 1.3095, test loss 1.4331
epoch  2: train loss 1.2703, test loss 1.4028
epoch  3: train loss 1.2404, test loss 1.3754
epoch  4: train loss 1.2382, test loss 1.3631
epoch  5: train loss 1.2157, test loss 1.3591
epoch  6: train loss 1.2088, test loss 1.3505
epoch  7: train loss 1.2071, test loss 1.3348
epoch  8: train loss 1.1915, test loss 1.3374
epoch  9: train loss 1.1929, test loss 1.3267


In [ ]:
context = torch.tensor(tokenizer.encode('def'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

In [ ]:
plt.plot(torch.tensor(l).view(-1, 10).mean(dim=-1))